# Assignment 11: Fine-Tuning GPT-2 on a Custom Text Style

**Name:** Ashif Hussain

**Reg no.:** 2411021240056

**Course:** Bachelor of Computer Application

**Semester:** V

**Subject:** Generative AI

**Faculty Name:** Sanjivini Chowdhury

**University:** Alliance University, Bengaluru

In [1]:
# Step 1: Install the tools we need (transformers, datasets, accelerate)
!pip install -q transformers datasets accelerate

In [2]:
# Step 2: Load the ORIGINAL GPT-2 and generate text (the "before")
from transformers import GPT2LMHeadModel, GPT2Tokenizer, set_seed

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
base_model = GPT2LMHeadModel.from_pretrained("gpt2")

# Same prompt as Assignment 2
prompt = "The scientist opened the door and saw"
input_ids = tokenizer.encode(prompt, return_tensors="pt")

# Fixed seed so the comparison is fair later
set_seed(42)

output = base_model.generate(
    input_ids,
    max_length=60,
    do_sample=True,
    top_p=0.9,
    pad_token_id=tokenizer.eos_token_id
)

base_text = tokenizer.decode(output[0], skip_special_tokens=True)
print(base_text)

# Save the output in a file for submission
with open("base_output.txt", "w") as f:
    f.write(base_text)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

The scientist opened the door and saw her daughter holding a knife.

"You can call us anything you want," she said, standing up, "but I'm not going anywhere. Go somewhere you can go. Call me Mom. We're here for you. That's it. Call us


In [3]:
# Step 3: Load dataset.txt and turn the words into numbers (tokens)
from datasets import load_dataset

# GPT-2 needs a padding token, so we reuse the end-of-text token
tokenizer.pad_token = tokenizer.eos_token

# Load our file: one line = one example
dataset = load_dataset("text", data_files={"train": "/content/sample_data/dataset.txt"})

# Remove any empty lines (cleaning)
dataset = dataset.filter(lambda x: len(x["text"].strip()) > 0)

# Turn each line into tokens
def tokenize(batch):
    return tokenizer(
        [t.strip() + tokenizer.eos_token for t in batch["text"]],
        truncation=True,
        max_length=64
    )

tokenized_dataset = dataset.map(tokenize, batched=True, remove_columns=["text"])

print(tokenized_dataset)
print("Number of lines:", len(tokenized_dataset["train"]))

Generating train split: 0 examples [00:00, ? examples/s]

Filter:   0%|          | 0/207 [00:00<?, ? examples/s]

Map:   0%|          | 0/207 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 207
    })
})
Number of lines: 207


In [4]:
# Step 4: Fine-tune GPT-2 on our dataset using the Hugging Face Trainer
from transformers import (
    GPT2LMHeadModel,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
)

# Start again from a fresh original GPT-2
finetune_model = GPT2LMHeadModel.from_pretrained("gpt2")

# This helper prepares the batches (mlm=False means normal GPT-style text)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# Training settings
training_args = TrainingArguments(
    output_dir="gpt2-shakespeare",
    num_train_epochs=8,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    logging_steps=20,
    save_strategy="no",
    report_to="none",
    fp16=True,
)

trainer = Trainer(
    model=finetune_model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    data_collator=data_collator,
)

# Start training
trainer.train()

# Save the fine-tuned model and tokenizer
trainer.save_model("gpt2-shakespeare")
tokenizer.save_pretrained("gpt2-shakespeare")
print("Training finished and model saved!")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
20,4.302586
40,3.829420
60,3.467284
80,3.167205
100,2.952195
120,2.733175
140,2.401182
160,2.585417
180,2.141025
200,2.094937


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training finished and model saved!


In [5]:
# Step 5: Generate text with the FINE-TUNED model (the "after")
from transformers import set_seed

# Same prompt as Assignment 2
prompt = "The scientist opened the door and saw"
input_ids = tokenizer.encode(prompt, return_tensors="pt").to(finetune_model.device)

# Same seed and same settings as the base model, so the comparison is fair
set_seed(42)

output = finetune_model.generate(
    input_ids,
    max_length=60,
    do_sample=True,
    top_p=0.9,
    pad_token_id=tokenizer.eos_token_id
)

finetuned_text = tokenizer.decode(output[0], skip_special_tokens=True)
print(finetuned_text)

# Save the output in a file for submission
with open("finetuned_output.txt", "w") as f:
    f.write(finetuned_text)

The scientist opened the door and saw a face shining like a moon. A smile lingered upon his face.

"What a wondrous name it is!" it called itself.


Then it was done with a single bow.

It never felt more like home.


In [6]:
# Step 6: Print both outputs side by side
print("=== BASE GPT-2 (before) ===")
print(base_text)
print()
print("=== FINE-TUNED GPT-2 (after) ===")
print(finetuned_text)

=== BASE GPT-2 (before) ===
The scientist opened the door and saw her daughter holding a knife.

"You can call us anything you want," she said, standing up, "but I'm not going anywhere. Go somewhere you can go. Call me Mom. We're here for you. That's it. Call us

=== FINE-TUNED GPT-2 (after) ===
The scientist opened the door and saw a face shining like a moon. A smile lingered upon his face.

"What a wondrous name it is!" it called itself.


Then it was done with a single bow.

It never felt more like home.


In [7]:
# Step 7: Save the before-and-after comparison in a file
comparison = f"""BEFORE-AND-AFTER COMPARISON
Prompt: The scientist opened the door and saw

=== BASE GPT-2 (before) ===
{base_text}

=== FINE-TUNED GPT-2 (after) ===
{finetuned_text}

DIFFERENCES IN STYLE
1. Word choice: The base model uses plain modern words (knife, I'm not going anywhere). The fine-tuned model uses poetic, old-style words (wondrous, lingered upon, shining like a moon).
2. Mood: The base model is tense and dangerous (a knife, an argument). The fine-tuned model is calm and gentle (a smile, never felt more like home).
3. Sentence style: The base model writes everyday speech. The fine-tuned model uses exclamations and short poetic lines like Shakespeare's plays.
"""

with open("comparison.txt", "w") as f:
    f.write(comparison)

print("comparison.txt saved!")

comparison.txt saved!


In [8]:
# Step 8: Download the output files to your computer
from google.colab import files

files.download("base_output.txt")
files.download("finetuned_output.txt")
files.download("comparison.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Analysis: Base GPT-2 vs Fine-Tuned GPT-2

**Prompt used:** "The scientist opened the door and saw"

**Differences in style:**

1. **Word choice:** The base model uses plain modern words (knife, "I'm not going anywhere"). The fine-tuned model uses poetic, old-style words (wondrous, lingered upon, shining like a moon).
2. **Mood:** The base model is tense and dangerous (a knife, an argument). The fine-tuned model is calm and gentle (a smile, "never felt more like home").
3. **Sentence style:** The base model writes everyday speech. The fine-tuned model uses exclamations and short poetic lines like Shakespeare's plays.

**Conclusion:** Fine-tuning on about 200 Shakespeare-style lines changed how GPT-2 writes. The prompt was the same, but the fine-tuned model followed the style of my dataset.